# XLM-R emotion training for Hindi-English and English-French

Start with Runtime > Change runtime type > GPU. Run cells in order or Run all,
responding to the upload and Drive prompts. This is task fine-tuning, not
training a language model from scratch. The provided bootstrap is synthetic;
its scores do not establish accuracy on naturally written or human-reviewed text.


In [ ]:
from pathlib import Path
import hashlib, json, zipfile, subprocess, sys, os
from google.colab import files
uploaded = files.upload()  # Select xlmr_multilingual.zip
archives = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(archives) == 1, "Upload exactly one training ZIP"
base = Path("/content").resolve()
with zipfile.ZipFile(archives[0]) as archive:
    assert sum(item.file_size for item in archive.infolist()) < 128 * 1024**2
    for item in archive.infolist():
        assert "\\" not in item.filename
        assert (base / item.filename).resolve().is_relative_to(base)
        assert (item.external_attr >> 16) & 0o170000 != 0o120000
    archive.extractall(base)
PACKAGE = base / "xlmr_multilingual_training"
manifest = json.loads((PACKAGE / "MANIFEST.json").read_text())
for name, expected in manifest["sha256"].items():
    path = (PACKAGE / name).resolve()
    assert path.is_relative_to(PACKAGE)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name
PROJECT = PACKAGE / "source"
os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[model]"], check=True)
sys.path.insert(0, str(PROJECT))
import torch
assert torch.cuda.is_available(), "Select a GPU runtime, then restart from the first cell"
print("GPU:", torch.cuda.get_device_name(0))


## Choose the experiment

`combined` trains one checkpoint on both language mixes. `separate` trains two
XLM-R checkpoints sequentially. Leave the defaults for the first run.
The original installed Hindi-only model is preserved on your laptop.

To use reviewed data, set USE_OWN_DATA=True and upload a canonical CSV containing
text,emotion,language_pair, optionally group_id and split. Use hi-en and fr-en.
Each language needs all seven emotions and enough independent groups per split.
The test split must remain untouched while choosing training settings.


In [ ]:
MODE = "combined"  # "combined" or "separate"
USE_OWN_DATA = False
ALLOW_SYNTHETIC = not USE_OWN_DATA
EPOCHS = 3  # Total target epochs, including already completed epochs on resume
DATA = PACKAGE / "data" / "all_splits.csv"
AUDIT = PACKAGE / "data" / "dataset_audit.json"
if USE_OWN_DATA:
    custom = files.upload()
    csvs = [name for name in custom if name.endswith(".csv")]
    assert len(csvs) == 1, "Upload one labelled canonical CSV"
    DATA = Path(csvs[0]).resolve()
    AUDIT = None
    # For a source requiring its matching audit, upload the audit JSON too.
    audits = [name for name in custom if name.endswith(".json")]
    if audits:
        assert len(audits) == 1
        AUDIT = Path(audits[0]).resolve()
assert MODE in {"combined", "separate"}
from google.colab import drive
drive.mount("/content/drive")
STORAGE = Path("/content/drive/MyDrive/CodeMood_XLMR_v2")
STORAGE.mkdir(parents=True, exist_ok=True)


## Validate and train

The bootstrap contains 16,000 training, 2,000 validation and 2,000 synthetic test
rows across the two languages. Each language contributes 8,000/1,000/1,000.
The same frozen split is retained after disconnection. Checkpoints go to Drive.
Batch size 2 and accumulation 8 usually give 16 phrases per optimizer update.
The best validation macro-F1 checkpoint is used for test evaluation.


In [ ]:
from training.data_workflow import prepare_data
jobs = [("all", "xlmr_multilingual_emotion")] if MODE == "combined" else [("hi-en", "xlmr_hi_en_emotion"), ("fr-en", "xlmr_fr_en_emotion")]
trained = []
for language, export_name in jobs:
    run = STORAGE / MODE / export_name
    manifest_path = run / "frozen_split.csv"
    prepared = prepare_data(DATA, "emotion", "raw", 42, manifest_path, AUDIT, language, allow_synthetic=ALLOW_SYNTHETIC)
    print(export_name, "split sizes:", [len(part) for part in prepared["parts"]])
    if not USE_OWN_DATA:
        expected = [16000, 2000, 2000] if language == "all" else [8000, 1000, 1000]
        assert [len(part) for part in prepared["parts"]] == expected
    command = [sys.executable, "-u", "-m", "training.compare_encoders", "--task", "emotion", "--models", "xlmr", "--data", str(DATA), "--language-pair", language, "--output-root", str(run / "models"), "--split-manifest", str(manifest_path), "--epochs", str(EPOCHS), "--batch-size", "2", "--gradient-accumulation", "8", "--max-length", "128", "--learning-rate", "2e-5", "--precision", "auto", "--text-mode", "raw", "--seed", "42", "--split-seed", "42", "--resume"]
    if AUDIT: command += ["--data-audit", str(AUDIT)]
    if ALLOW_SYNTHETIC: command += ["--allow-synthetic"]
    subprocess.run(command, cwd=PROJECT, check=True)
    trained.append((run / "models" / "xlmr_emotion", export_name))


## Inspect results and export

Report per-language and per-emotion support, accuracy, macro-F1 and errors.
Bootstrap scores are synthetic-only. Do not compare these seven-class results
directly with Team 5 G2's three-class results. Real improvement needs the same
independent test labels and comparable tasks.


In [ ]:
import pandas as pd
from training.cloud_bundle import export_checkpoint
exports = []
for model_dir, export_name in trained:
    metrics = json.loads((model_dir / "metrics.json").read_text())
    print(export_name, "scope:", metrics["evaluation_scope"])
    display(pd.DataFrame(metrics["test_by_language"]).T)
    display(pd.DataFrame(metrics["test"]["per_class"]).T)
    display(pd.DataFrame(metrics["history"]))
    output = export_checkpoint(model_dir, STORAGE / f"{export_name}.zip")
    exports.append(output)
    print("Saved inference export:", output)
for output in exports:
    files.download(str(output))


## Install on the laptop

From the personal project root, run:
`python -m training.install_checkpoint "C:/path/to/downloaded/xlmr_multilingual_emotion.zip"`

For separate runs, install each ZIP the same way. The installer selects the
language-specific folder from metadata, rejects incomplete/unsafe exports and
preserves existing models. Restart or refresh the Streamlit app. XLM-R routes
to a checkpoint trained for the selected language. The ZIP excludes optimizer
state and raw training phrases. Keep Drive recovery files if resuming training.
